In [ ]:
!pip install -q keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import tensorflow as tf
import keras_tuner as kt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
data = fetch_california_housing()

X = data.data
y = data.target

print(X.shape)
print(y.shape)

(20640, 8)
(20640,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
def build_regression_model(hp):
    model = tf.keras.Sequential()

    model.add(tf.keras.layers.Input(shape=(X_train.shape[1],)))

    for i in range(hp.Int("num_layers", 1, 3)):
        model.add(
            tf.keras.layers.Dense(
                units=hp.Int(f"units_{i}", 32, 128, step=32),
                activation="relu"
            )
        )

        model.add(
            tf.keras.layers.Dropout(
                hp.Float("dropout", 0.0, 0.5, step=0.1)
            )
        )

    optimizer_name = hp.Choice("optimizer", ["adam", "sgd"])

    learning_rate = hp.Choice(
        "learning_rate",
        [0.001, 0.01, 0.0001]
    )

    if optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        )
    else:
        optimizer = tf.keras.optimizers.SGD(
            learning_rate=learning_rate
        )

    model.add(tf.keras.layers.Dense(1))

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mse"]
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_regression_model,
    objective="val_loss",
    max_trials=10,
    directory="regression_tuning",
    project_name="california_housing"
)

In [ ]:
tuner.search(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    verbose=1
)

Trial 10 Complete [00h 00m 48s]
val_loss: 0.35423728823661804

Best val_loss So Far: 0.28558436036109924
Total elapsed time: 00h 09m 54s


In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print("Number of Layers:", best_hp.get("num_layers"))
print("Dropout:", best_hp.get("dropout"))
print("Optimizer:", best_hp.get("optimizer"))
print("Learning Rate:", best_hp.get("learning_rate"))

for i in range(best_hp.get("num_layers")):
    print(f"Units Layer {i+1}:", best_hp.get(f"units_{i}"))

Number of Layers: 2
Dropout: 0.0
Optimizer: adam
Learning Rate: 0.01
Units Layer 1: 96
Units Layer 2: 64


In [ ]:
best_model = tuner.hypermodel.build(best_hp)


In [ ]:
history = best_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1
)

Epoch 1/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.6408 - mse: 0.6408 - val_loss: 0.4319 - val_mse: 0.4319
Epoch 2/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.3701 - mse: 0.3701 - val_loss: 0.4138 - val_mse: 0.4138
Epoch 3/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.3745 - mse: 0.3745 - val_loss: 0.3536 - val_mse: 0.3536
Epoch 4/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.3402 - mse: 0.3402 - val_loss: 0.3594 - val_mse: 0.3594
Epoch 5/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3383 - mse: 0.3383 - val_loss: 0.3496 - val_mse: 0.3496
Epoch 6/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3522 - mse: 0.3522 - val_loss: 0.3636 - val_mse: 0.3636
Epoch 7/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3184 - mse: 0.3184 - val_loss: 0.3296 - val_mse: 0.3296
Epoch 8/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3147 - mse: 0.3147 - val_loss: 0.3194 - val_mse: 0.3194
Epoch 9/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - lo

In [12]:
y_pred = best_model.predict(X_test).flatten()

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
MSE: 0.29307311582382894
RMSE: 0.5413622777990991
R2 Score: 0.776349887180068


In [13]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

iris = load_iris()

X = iris.data
y = to_categorical(iris.target, 3)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
def build_model(hp):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(4,)),
        tf.keras.layers.Dense(
            hp.Int("units", 16, 64, step=16),
            activation="relu"
        ),
        tf.keras.layers.Dropout(
            hp.Float("dropout", 0, 0.5, step=0.25)
        ),
        tf.keras.layers.Dense(3, activation="softmax")
    ])

    lr = hp.Choice("learning_rate", [0.001, 0.01])
    opt = hp.Choice("optimizer", ["adam", "sgd"])

    optimizer = tf.keras.optimizers.Adam(lr) if opt == "adam" else tf.keras.optimizers.SGD(lr)

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [15]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    directory="tuner",
    project_name="iris_mlp"
)

tuner.search(
    X_train,
    y_train,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

Trial 5 Complete [00h 00m 04s]
val_accuracy: 0.5833333134651184

Best val_accuracy So Far: 0.9583333134651184
Total elapsed time: 00h 00m 27s


In [16]:
model = tuner.get_best_models(1)[0]

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Test Loss: 0.2557258903980255
Test Accuracy: 0.8666666746139526
